In [ ]:
# CELL 1: Environment setup (v8 = v7 + my augmented data)
import os, sys
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import glob, subprocess, site, shutil, stat

# Install Triton wheel
candidates = glob.glob('/kaggle/input/**/*triton*.whl', recursive=True)
print('Found Triton wheels:', candidates)
wheel = candidates[0]
target = '/kaggle/working/pydeps'
os.makedirs(target, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install',
    '--no-deps', '--target', target, '--upgrade', '--ignore-installed', wheel], check=True)
if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)
print('Triton installed!')

# Fix ptxas for Blackwell
sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
ptxas_dst = '/tmp/ptxas-blackwell'
if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
    shutil.copy2(ptxas_src, ptxas_dst)
    os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    src_bin = os.path.dirname(ptxas_src)
    dst_bin = '/tmp/triton_nvidia_bin'
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
    import triton.backends.nvidia as nv_backend
    nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
    os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
import triton.backends.nvidia.compiler as nv_compiler
nv_compiler.get_ptxas_version = lambda arch: '12.0'
print('Blackwell ptxas fixed!')

# Install Unsloth + deps offline to a writable target dir
packages_dir = '/kaggle/input/datasets/mayukh18/nemotron-packages/packages'
INSTALL_DIR = '/kaggle/working/site-packages'
os.makedirs(INSTALL_DIR, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    '--no-index', '--find-links', packages_dir, '--target', INSTALL_DIR,
    'unsloth', 'trl', 'peft', 'transformers', 'datasets', 'accelerate', 'bitsandbytes'],
    check=False)
if INSTALL_DIR not in sys.path:
    sys.path.insert(0, INSTALL_DIR)
site.addsitedir(INSTALL_DIR)

all_mamba = sorted(glob.glob('/kaggle/input/**/mamba_ssm-*.whl', recursive=True))
all_causal = sorted(glob.glob('/kaggle/input/**/causal*conv1d*.whl', recursive=True))
if all_causal:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--target', INSTALL_DIR, all_causal[-1]], check=False)
if all_mamba:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--target', INSTALL_DIR, all_mamba[-1]], check=False)
print('All packages installed!')

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Cell 1 done!')

In [ ]:
# CELL 2: Load model with Unsloth + LoRA
import torch, kagglehub
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 8192
MODEL_PATH = '/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    unsloth_force_compile=False,
    attn_implementation='eager',
    dtype=torch.bfloat16,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Model loaded with Unsloth.')

# Apply LoRA - NO lm_head (key difference!)
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                  'in_proj', 'out_proj', 'up_proj', 'down_proj']

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=target_modules,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()
print('Cell 2 done!')

In [ ]:
# CELL 3: Train with SFTTrainer + stratified batching
import pandas as pd, random, gc, time, re, math
from collections import defaultdict
from datasets import Dataset as HFDataset
from torch.utils.data import DataLoader, Sampler
from trl import SFTTrainer, SFTConfig

SEED = 42
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

# Load dgxchen dataset
DATASET_PATH = '/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv'
df = pd.read_csv(DATASET_PATH)
print(f'dgxchen rows: {len(df)}')

# Load MY augmented examples (gravity / unit_conversion / numeral),
# every answer independently verified, no rounding contradictions.
AUG_PATH = '/kaggle/input/datasets/REPLACE_AUG/augmented_examples.csv'
import os as _os
if _os.path.exists(AUG_PATH):
    aug = pd.read_csv(AUG_PATH)
    print(f'augmented rows: {len(aug)}')
    df = pd.concat([df, aug], ignore_index=True)
    print(f'combined rows: {len(df)}')
else:
    print('WARNING: augmented file not found, training on dgxchen only')

train_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

records = []
record_types = []
for _, row in train_df.iterrows():
    prompt = str(row['prompt'])
    answer = str(row['answer'])
    cot = str(row['generated_cot'])
    if not cot or cot == 'nan' or len(cot.strip()) < 5:
        continue
    cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
    user_content = prompt + PROMPT_SUFFIX
    assistant_content = cot_cleaned + f'\n</think>\n\\boxed{{{answer}}}'
    records.append({'messages': [
        {'role': 'user', 'content': user_content},
        {'role': 'assistant', 'content': assistant_content},
    ]})
    record_types.append(str(row['type']))

dataset = HFDataset.from_list(records)
print(f'SFT records: {len(records)}')

def formatting_prompts_func(example):
    messages = example['messages']
    if messages and isinstance(messages[0], dict):
        conversations = [messages]
    else:
        conversations = messages
    texts = []
    for conversation in conversations:
        try:
            text = tokenizer.apply_chat_template(
                conversation, tokenize=False,
                add_generation_prompt=False, enable_thinking=True)
        except TypeError:
            text = tokenizer.apply_chat_template(
                conversation, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return texts

training_args = SFTConfig(
    output_dir='/kaggle/working/sft_output',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=2e-4,
    lr_scheduler_type='linear',
    warmup_steps=0,
    max_length=8192,
    adam_beta1=0.9,
    adam_beta2=0.95,
    adam_epsilon=1e-8,
    weight_decay=0.0,
    max_grad_norm=1e9,
    logging_steps=10,
    save_strategy='no',
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=2,
    remove_unused_columns=False,
    seed=SEED,
    report_to='none',
    packing=False,
)

def build_stratified_index_order(labels, batch_size, seed):
    by_label = defaultdict(list)
    for idx, label in enumerate(labels):
        by_label[label].append(idx)
    rng = random.Random(seed)
    for idx_list in by_label.values():
        rng.shuffle(idx_list)
    n_batches = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(n_batches)]
    batch_order = list(range(n_batches))
    rng.shuffle(batch_order)
    assigned = 0
    for label in sorted(by_label.keys()):
        for idx in by_label[label]:
            batches[batch_order[assigned % n_batches]].append(idx)
            assigned += 1
    order = [idx for batch in batches for idx in batch]
    return order

class PrecomputedOrderSampler(Sampler):
    def __init__(self, order): self.order = list(order)
    def __iter__(self): return iter(self.order)
    def __len__(self): return len(self.order)

class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_order=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.stratified_order = stratified_order
    def get_train_dataloader(self):
        if self.stratified_order is None:
            return super().get_train_dataloader()
        dataloader_kwargs = {
            'batch_size': self.args.per_device_train_batch_size,
            'sampler': PrecomputedOrderSampler(self.stratified_order),
            'collate_fn': self.data_collator,
            'num_workers': self.args.dataloader_num_workers,
            'pin_memory': self.args.dataloader_pin_memory,
            'persistent_workers': self.args.dataloader_persistent_workers,
            'drop_last': self.args.dataloader_drop_last,
        }
        if self.args.dataloader_num_workers > 0:
            dataloader_kwargs['prefetch_factor'] = self.args.dataloader_prefetch_factor
        return DataLoader(self.train_dataset, **dataloader_kwargs)

effective_batch_size = max(1, training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
stratified_order = build_stratified_index_order(record_types, effective_batch_size, SEED)
print(f'Effective batch size: {effective_batch_size}')

trainer = StratifiedSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    formatting_func=formatting_prompts_func,
    stratified_order=stratified_order,
)

print('Starting SFT training...')
t0 = time.time()
trainer.train()
print(f'Training done in {(time.time()-t0)/60:.1f} min')

ADAPTER_DIR = '/kaggle/working/sft_adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved to {ADAPTER_DIR}')
print('Cell 3 done!')

In [ ]:
# CELL 4: Create submission.zip
import json, os, shutil, zipfile

BASE_MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
OUTPUT_DIR = '/kaggle/working'
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, 'submission_adapter')
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ['adapter_config.json', 'adapter_model.safetensors']
src_adapter_dir = '/kaggle/working/sft_adapter'

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    shutil.copy2(src, dst)
    print(f'Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)')

# Fix config
config_path = os.path.join(SUBMISSION_ADAPTER_DIR, 'adapter_config.json')
with open(config_path, 'r') as f:
    cfg = json.load(f)
cfg['base_model_name_or_path'] = BASE_MODEL_NAME
cfg['inference_mode'] = True
cfg['lora_dropout'] = 0.0
with open(config_path, 'w') as f:
    json.dump(cfg, f, indent=2)

# Create zip
zip_path = os.path.join(OUTPUT_DIR, 'submission.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f'Added {fname}')

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f'\nsubmission.zip: {zip_sz:.1f} MB')
print('Done! Ready to submit.')